# 00 — Setup & Exploração do FTP

**Objetivo:** validar acesso ao FTP do Novo CAGED, baixar **1 mês piloto**, confirmar delimitador/encoding e o **código de município** (6 dígitos) antes de processar a janela completa.

> Pré-requisito: `docker compose up -d db` na raiz e `cp .env.example .env` aqui.

In [ ]:
import sys, pathlib
# garante que o pacote caged_etl (em ../) seja importável a partir do notebook
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from caged_etl import setup_logging, get_settings
setup_logging()
settings = get_settings()
print('Municípios:', settings.municipios, '| Anos:', settings.ano_inicio, '-', settings.ano_fim)

## 1. Conexão e listagem de uma competência

In [ ]:
from caged_etl import ftp_client

PILOTO = '202401'  # mês piloto
arquivos = ftp_client.list_remote_files(PILOTO)
arquivos

## 2. Download do CAGEDMOV piloto

In [ ]:
zip_path = ftp_client.download_mov(PILOTO)
zip_path

## 3. Extração do .7z

In [ ]:
from caged_etl import extract
txts = extract.extract_file(zip_path)
txts

## 4. Encoding, delimitador e cabeçalho

In [ ]:
import polars as pl
from caged_etl.raw import detect_encoding

txt = txts[0]
enc = detect_encoding(txt)
print('encoding detectado:', enc)

preview = pl.read_csv(txt, separator=';', encoding=enc, infer_schema_length=0, n_rows=5)
print('colunas originais:', preview.columns)
preview

## 5. Validação do código de município (6 dígitos)
Confirma que `270430` (Maceió) e `270030` (Arapiraca) existem na coluna `município`.

In [ ]:
from caged_etl.config import normalize_colname
col_mun = next(c for c in preview.columns if normalize_colname(c) == 'municipio')
amostra = pl.read_csv(txt, separator=';', encoding=enc, infer_schema_length=0,
                      columns=[col_mun])
vc = amostra[col_mun].value_counts().sort('count', descending=True)
print('exemplos de códigos de município:')
print(vc.head(10))
alvo = {str(m) for m in settings.municipios}
encontrados = set(amostra[col_mun].unique().to_list()) & alvo
print('municípios-alvo encontrados:', encontrados)
assert encontrados == alvo, 'Conferir formato do código de município no layout!'

## ✅ Conclusões
- FTP acessível, arquivo `CAGEDMOV` baixado e extraído.
- Delimitador `;` e encoding confirmados.
- Códigos de município de 6 dígitos validados → seguir para o `01`.